# TRÍCH XUẤT BẢNG TỪ ẢNH TÀI LIỆU
### Pipeline: Table Structure Recognition + VietOCR (`vgg_transformer`)

- **Mô hình cấu trúc bảng**: `microsoft/table-transformer-structure-recognition-v1.1-all` (kết hợp với hình thái học đường lưới cho bảng kẻ viền).
- **Mô hình OCR tiếng Việt**: VietOCR (`vgg_transformer`).
- **Tính năng nâng cao**: Xử lý ô gộp ngang `[[H]]`, gộp dọc `[[V]]`, xuống dòng trong ô `<br>`, in đậm `**...**` cho header/banner, và escape ký tự pipe `\|`.


## 1. Cấu hình môi trường & đường dẫn


In [ ]:
import os
from pathlib import Path
import time
import zipfile
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np

# Tự động nhận diện thư mục làm việc
cwd = Path.cwd()
if cwd.name == 'tacvu4_table_extraction':
    ROOT = cwd
else:
    ROOT = cwd / 'tacvu4_table_extraction'

PROJECT_ROOT = ROOT.parent
DATA = ROOT / 'data'
TRAIN_DIR = DATA / 'training_set'
RUNS = ROOT / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)

# Chọn tập dữ liệu dự đoán: 'public_test' hoặc 'private_test'
SPLIT = 'private_test'
PRIVATE_PASSWORD = '225554'

# Tự động kiểm tra và giải nén private_test.zip nếu chưa giải nén
private_zip = DATA / 'private_test.zip'
private_dir = DATA / 'private_test'
if SPLIT == 'private_test' and private_zip.exists() and (not private_dir.exists() or not (private_dir / 'manifest.jsonl').exists()):
    print(f'[giải nén] Đang giải nén {private_zip} với mật khẩu...')
    with zipfile.ZipFile(private_zip, 'r') as zf:
        zf.extractall(DATA, pwd=PRIVATE_PASSWORD.encode('utf-8'))
    print(f'[giải nén] Hoàn tất giải nén vào {private_dir}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TABLE_MODEL_PATH = PROJECT_ROOT / 'models' / 'table_microsoft'

print(f'[cấu hình] ROOT: {ROOT}')
print(f'[cấu hình] SPLIT: {SPLIT}')
print(f'[cấu hình] Thiết bị: {DEVICE}')
print(f'[cấu hình] Table Transformer model: {TABLE_MODEL_PATH}')


## 2. Tiện ích dữ liệu & Định dạng Markdown chuẩn


In [ ]:
import json
import unicodedata

EMPTY_TABLE = '| A | B |\n| --- | --- |\n| x | y |\n'

def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def load_manifest(split_dir: Path) -> list[dict]:
    path = split_dir / 'manifest.jsonl'
    if not path.is_file():
        raise FileNotFoundError(f'Không thấy manifest: {path}')
    return load_jsonl(path)

def split_markdown_row(line: str) -> list[str]:
    if not (line.startswith('|') and line.endswith('|')):
        raise ValueError('Dòng Markdown không bắt đầu/kết thúc bằng dấu |')
    cells: list[str] = []
    current: list[str] = []
    escaped = False
    for character in line[1:-1]:
        if escaped:
            current.append('\\' + character)
            escaped = False
        elif character == '\\':
            escaped = True
        elif character == '|':
            cells.append(''.join(current).strip())
            current = []
        else:
            current.append(character)
    if escaped:
        current.append('\\')
    cells.append(''.join(current).strip())
    return cells

def parse_markdown(markdown: str) -> list[list[list[str]]]:
    tables: list[list[list[str]]] = []
    for block in unicodedata.normalize('NFC', markdown).strip().split('\n\n'):
        lines = [line.strip() for line in block.splitlines() if line.strip()]
        if len(lines) < 3:
            raise ValueError('Bảng có ít hơn ba dòng')
        rows = [split_markdown_row(line) for line in lines]
        width = len(rows[0])
        if width < 2 or any(len(row) != width for row in rows):
            raise ValueError('Số ô giữa các dòng không nhất quán')
        if any(cell != '---' for cell in rows[1]):
            raise ValueError('Thiếu dòng phân cách Markdown')
        tables.append([rows[0], *rows[2:]])
    if not tables:
        raise ValueError('Không có bảng')
    return tables

def is_valid_markdown(markdown: str) -> bool:
    try:
        parse_markdown(markdown)
        return '```' not in markdown
    except (ValueError, IndexError):
        return False

def write_predictions(output_dir: Path, records: list[dict], markdowns: list[str]) -> None:
    if len(records) != len(markdowns):
        raise ValueError('Số prediction không khớp manifest')
    output_dir.mkdir(parents=True, exist_ok=True)
    expected = {f"{record['id']}.md" for record in records}
    for existing in output_dir.glob('*.md'):
        if existing.name not in expected:
            existing.unlink()
    for record, markdown in zip(records, markdowns):
        normalized = unicodedata.normalize('NFC', markdown).strip() + '\n'
        if not is_valid_markdown(normalized):
            # Dự phòng bảng hợp lệ nếu phát sinh lỗi cú pháp ngoại lệ
            normalized = EMPTY_TABLE
        (output_dir / f"{record['id']}.md").write_text(normalized, encoding='utf-8')

def make_predictions_zip(output_dir: Path, zip_path: Path) -> None:
    files = sorted(output_dir.glob('*.md'))
    if not files:
        raise ValueError('Thư mục prediction rỗng')
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in files:
            archive.write(path, path.name)


## 3. Khởi tạo 2 Mô hình: Table Transformer & VietOCR


In [ ]:
from transformers import TableTransformerForObjectDetection, DetrImageProcessor
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

print('[1/2] Đang nạp mô hình Table Transformer (Structure Recognition)...')
processor = DetrImageProcessor(
    do_resize=True,
    size={'shortest_edge': 800, 'longest_edge': 1000},
    do_rescale=True,
    do_normalize=True
)
table_transformer = TableTransformerForObjectDetection.from_pretrained(str(TABLE_MODEL_PATH))
table_transformer.to(DEVICE)
table_transformer.eval()
print('  -> Table Transformer sẵn sàng!')

print('[2/2] Đang nạp mô hình VietOCR (vgg_transformer)...')
ocr_config = Cfg.load_config_from_name('vgg_transformer')
ocr_config['device'] = 'cpu'  # CPU trên Mac chạy rất nhanh và ổn định
ocr_predictor = Predictor(ocr_config)
print('  -> VietOCR sẵn sàng!')


## 4. Thuật toán định vị lưới & Nhận diện ô gộp ([[H]], [[V]])


In [ ]:
from dataclasses import dataclass
import cv2

@dataclass(frozen=True)
class GridTable:
    bbox: tuple[int, int, int, int]
    x_edges: tuple[int, ...]
    y_edges: tuple[int, ...]

    @property
    def rows(self) -> int:
        return max(0, len(self.y_edges) - 1)

    @property
    def cols(self) -> int:
        return max(0, len(self.x_edges) - 1)

def _runs(mask: np.ndarray, minimum: int = 1, gap: int = 2) -> list[tuple[int, int]]:
    indices = np.flatnonzero(mask)
    if indices.size == 0:
        return []
    result: list[tuple[int, int]] = []
    start = previous = int(indices[0])
    for raw in indices[1:]:
        value = int(raw)
        if value - previous > gap:
            if previous - start + 1 >= minimum:
                result.append((start, previous))
            start = value
        previous = value
    if previous - start + 1 >= minimum:
        result.append((start, previous))
    return result

def load_gray(path: Path) -> np.ndarray:
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f'Không đọc được ảnh: {path}')
    return gray

def deskew_gray(gray: np.ndarray, maximum_degrees: float = 4.0) -> tuple[np.ndarray, float]:
    edges = cv2.Canny(gray, 60, 180)
    height, width = gray.shape
    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 1800,
        threshold=max(70, width // 12),
        minLineLength=max(180, width // 5),
        maxLineGap=max(12, width // 80),
    )
    angles: list[float] = []
    if lines is not None:
        for raw in np.asarray(lines).reshape(-1, 4):
            x0, y0, x1, y1 = map(float, raw)
            angle = float(np.degrees(np.arctan2(y1 - y0, x1 - x0)))
            while angle > 90:
                angle -= 180
            while angle < -90:
                angle += 180
            if abs(angle) <= maximum_degrees:
                angles.append(angle)
    if not angles:
        return gray, 0.0
    angle = float(np.median(angles))
    if abs(angle) < 0.08:
        return gray, 0.0
    matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
    rotated = cv2.warpAffine(
        gray,
        matrix,
        (width, height),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255,
    )
    return rotated, angle

def binarize(gray: np.ndarray) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    return cv2.threshold(clahe, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

def line_masks(binary: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    height, width = binary.shape
    horizontal = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (max(45, width // 18), 1)),
    )
    vertical = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(35, height // 45))),
    )
    return horizontal, vertical

def _line_positions(mask: np.ndarray, axis: int, extent: int) -> list[int]:
    projection = np.count_nonzero(mask, axis=axis)
    threshold = max(8, round(extent * 0.42))
    positions: list[int] = []
    for start, end in _runs(projection >= threshold, gap=3):
        if end - start >= 9:
            positions.extend((start, end))
        else:
            positions.append(round((start + end) / 2))
    return positions

def _dedupe(values: list[int], tolerance: int = 6) -> list[int]:
    if not values:
        return []
    groups = [[values[0]]]
    for value in values[1:]:
        if value - groups[-1][-1] <= tolerance:
            groups[-1].append(value)
        else:
            groups.append([value])
    return [round(sum(group) / len(group)) for group in groups]

def detect_grid_tables(gray: np.ndarray) -> list[GridTable]:
    binary = binarize(gray)
    horizontal, vertical = line_masks(binary)
    joined = cv2.dilate(
        cv2.bitwise_or(horizontal, vertical),
        cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)),
        iterations=1,
    )
    contours, _ = cv2.findContours(joined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    height, width = gray.shape
    candidates: list[GridTable] = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w < width * 0.48 or h < 80 or w * h < width * height * 0.012:
            continue
        pad = 4
        x0, y0 = max(0, x - pad), max(0, y - pad)
        x1, y1 = min(width, x + w + pad), min(height, y + h + pad)
        local_h = horizontal[y0:y1, x0:x1]
        local_v = vertical[y0:y1, x0:x1]
        xs = _line_positions(local_v, axis=0, extent=y1 - y0)
        ys = _line_positions(local_h, axis=1, extent=x1 - x0)
        xs = _dedupe([x0 + value for value in xs])
        ys = _dedupe([y0 + value for value in ys])
        if len(xs) < 3 or len(ys) < 3:
            continue
        if abs(xs[0] - x) > 10:
            xs.insert(0, x)
        if abs(xs[-1] - (x + w - 1)) > 10:
            xs.append(x + w - 1)
        if abs(ys[0] - y) > 10:
            ys.insert(0, y)
        if abs(ys[-1] - (y + h - 1)) > 10:
            ys.append(y + h - 1)
        if 2 <= len(xs) - 1 <= 18 and 2 <= len(ys) - 1 <= 60:
            candidates.append(GridTable((x, y, x + w, y + h), tuple(xs), tuple(ys)))

    candidates.sort(key=lambda item: (item.bbox[1], item.bbox[0], -item.rows * item.cols))
    kept: list[GridTable] = []
    for table in candidates:
        x0, y0, x1, y1 = table.bbox
        area = max(1, (x1 - x0) * (y1 - y0))
        duplicate = False
        for previous in kept:
            a0, b0, a1, b1 = previous.bbox
            intersection = max(0, min(x1, a1) - max(x0, a0)) * max(0, min(y1, b1) - max(y0, b0))
            if intersection / min(area, max(1, (a1 - a0) * (b1 - b0))) > 0.72:
                duplicate = True
                break
        if not duplicate:
            kept.append(table)
    return kept


## 5. Module Nhận diện Chữ Ô & Trích xuất Bảng Đầy Đủ


In [ ]:
from PIL import Image
import unicodedata
import re

def clean_ocr_text(text: str, is_dark_banner: bool = False) -> str:
    text = unicodedata.normalize('NFC', text).strip()
    text = text.replace('|', '\|')
    
    # Chuẩn hoá các mã tiền tố phổ biến trong bảng (VD: GTO4 -> GT04, CHO1 -> CH01)
    text = re.sub(r'^(GT|CH|HD|NV|KH|SP|VT|DA|DD)[Oo](\d+)', r'\g<1>0\2', text)
    text = re.sub(r'\b(GT|CH|HD|NV|KH|SP|VT|DA|DD)[Oo](\d+)', r'\g<1>0\2', text)
    
    if is_dark_banner:
        # Tiêu đề bảng hành chính luôn in hoa toàn bộ và có dấu gạch ngang dài em-dash
        text = text.replace('?', '—').replace('–', '—').replace('-', '—')
        text = re.sub(r'\s*—\s*', ' — ', text)
        text = text.upper()
        
    return text

def detect_row_boldness(gray: np.ndarray, table: GridTable) -> list[bool]:
    """Phát hiện in đậm: Dòng đầu (header) và dòng cuối (summary) luôn in đậm 100% trong M1 & M2."""
    bold_rows = []
    num_rows = table.rows
    for r in range(num_rows):
        if r == 0 or r == num_rows - 1:
            bold_rows.append(True)
            continue
            
        row_crop = gray[table.y_edges[r] : table.y_edges[r+1], table.x_edges[0] : table.x_edges[-1]]
        if np.mean(row_crop < 100) > 0.65:
            bold_rows.append(True)
            continue
            
        retentions = []
        for c in range(table.cols):
            crop = gray[table.y_edges[r] + 3 : table.y_edges[r + 1] - 3, table.x_edges[c] + 3 : table.x_edges[c + 1] - 3]
            if crop.size == 0 or crop.std() < 6.0:
                continue
            thresh = cv2.threshold(crop, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
            nz = np.count_nonzero(thresh)
            if nz > 25:
                eroded = cv2.erode(thresh, np.ones((2, 2), np.uint8))
                retentions.append(np.count_nonzero(eroded) / nz)
        avg_ret = np.mean(retentions) if retentions else 0.0
        bold_rows.append(avg_ret > 0.48)
    return bold_rows

def ocr_single_cell(cell_bgr: np.ndarray, is_bold: bool = False) -> str:
    if cell_bgr.size == 0 or cell_bgr.shape[0] < 6 or cell_bgr.shape[1] < 6:
        return ''
    gray = cv2.cvtColor(cell_bgr, cv2.COLOR_BGR2GRAY)
    
    # 1. Phát hiện banner / tiêu đề nền tối chữ sáng (Inverse Text)
    if np.mean(gray < 100) > 0.60:
        ink = (gray > 140)
        ys, xs = np.nonzero(ink)
        if len(xs) < 15:
            return ''
        y0, y1 = max(0, ys.min() - 4), min(gray.shape[0], ys.max() + 5)
        x0, x1 = max(0, xs.min() - 8), min(gray.shape[1], xs.max() + 9)
        tight = 255 - gray[y0:y1, x0:x1]
        pil = Image.fromarray(tight).convert('RGB')
        text = clean_ocr_text(ocr_predictor.predict(pil), is_dark_banner=True)
        return f'**{text}**' if text else ''
        
    # 2. Kiểm tra ô rỗng / nhiễu giấy scan (tránh ảo giác Contraction)
    inner = gray[4:-4, 4:-4]
    if inner.size == 0 or inner.std() < 6.0 or inner.min() > 185:
        return ''
        
    thresh = cv2.threshold(inner, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    nz = np.count_nonzero(thresh)
    if nz < 15:
        return ''
        
    # 3. Phát hiện ô nhiều dòng (<br>) bằng Run-based projection an toàn (yêu cầu khoảng trống giữa 2 dòng thực tế)
    if cell_bgr.shape[0] >= 50:
        proj = np.count_nonzero(thresh, axis=1)
        runs = []
        in_run = False
        start = 0
        for i, v in enumerate(proj):
            if v > 6 and not in_run:
                in_run = True
                start = i
            elif v <= 6 and in_run:
                in_run = False
                if i - start >= 7:
                    runs.append((start, i))
        if in_run and len(proj) - start >= 7:
            runs.append((start, len(proj)))
            
        if len(runs) >= 2:
            valley = proj[runs[0][1] : runs[1][0]]
            if len(valley) >= 3 and np.min(valley) <= 2:
                split_y = (runs[0][1] + runs[1][0]) // 2 + 4
                top_crop = Image.fromarray(cv2.cvtColor(cell_bgr[:split_y, :], cv2.COLOR_BGR2RGB))
                bot_crop = Image.fromarray(cv2.cvtColor(cell_bgr[split_y:, :], cv2.COLOR_BGR2RGB))
                t1 = clean_ocr_text(ocr_predictor.predict(top_crop))
                t2 = clean_ocr_text(ocr_predictor.predict(bot_crop))
                if t1 and t2:
                    content = f'{t1}<br>{t2}'
                else:
                    content = t1 or t2
                if is_bold and content:
                    return f'**{content}**'
                return content
                
    # 4. Ô 1 dòng thông thường: cắt khít theo bounding box mực thực để không biến dạng tỷ lệ khung hình
    ys, xs = np.nonzero(thresh)
    if len(xs) == 0:
        return ''
    y0 = max(0, 4 + ys.min() - 3)
    y1 = min(cell_bgr.shape[0], 4 + ys.max() + 4)
    x0 = max(0, 4 + xs.min() - 6)
    x1 = min(cell_bgr.shape[1], 4 + xs.max() + 7)
    tight = cell_bgr[y0:y1, x0:x1]
    
    pil_img = Image.fromarray(cv2.cvtColor(tight, cv2.COLOR_BGR2RGB))
    text = clean_ocr_text(ocr_predictor.predict(pil_img))
    if is_bold and text:
        text = f'**{text}**'
    return text

def extract_table_from_image(img_path: Path) -> str:
    orig_bgr = cv2.imread(str(img_path))
    if orig_bgr is None:
        return EMPTY_TABLE
    gray = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2GRAY)
    binary = binarize(gray)
    horizontal, vertical = line_masks(binary)
    tables = detect_grid_tables(gray)
    if not tables:
        return EMPTY_TABLE
        
    table_blocks = []
    for table in tables:
        num_rows, num_cols = table.rows, table.cols
        if num_rows < 2 or num_cols < 2:
            continue
            
        grid = [[None for _ in range(num_cols)] for _ in range(num_rows)]
        bold_rows = detect_row_boldness(gray, table)
        
        # Nhận diện hàng tiêu đề toàn dải nền đen/tối
        for r in range(num_rows):
            row_crop = gray[table.y_edges[r] : table.y_edges[r+1], table.x_edges[0] : table.x_edges[-1]]
            if np.mean(row_crop < 100) > 0.65:
                for c in range(1, num_cols):
                    grid[r][c] = '[[H]]'
        
        # Dò ô gộp dọc [[V]]
        for r in range(num_rows - 1):
            yr = table.y_edges[r + 1]
            for c in range(num_cols):
                x0, x1 = table.x_edges[c], table.x_edges[c + 1]
                v_slice = horizontal[max(0, yr - 3) : min(horizontal.shape[0], yr + 4), x0 + 4 : x1 - 4]
                if np.count_nonzero(v_slice) <= (x1 - x0) * 0.25:
                    grid[r + 1][c] = '[[V]]'
                    
        # Dò ô gộp ngang [[H]]
        for r in range(num_rows):
            y0, y1 = table.y_edges[r], table.y_edges[r + 1]
            for c in range(num_cols - 1):
                if grid[r][c] == '[[V]]' or grid[r][c+1] == '[[H]]':
                    continue
                xc = table.x_edges[c + 1]
                h_slice = vertical[y0 + 4 : y1 - 4, max(0, xc - 3) : min(vertical.shape[1], xc + 4)]
                if np.count_nonzero(h_slice) <= (y1 - y0) * 0.25:
                    grid[r][c + 1] = '[[H]]'
                    
        # Đọc chữ bằng VietOCR cho các ô gốc
        for r in range(num_rows):
            is_row_bold = bold_rows[r]
            for c in range(num_cols):
                if grid[r][c] in ['[[H]]', '[[V]]']:
                    continue
                span_c, span_r = 1, 1
                while c + span_c < num_cols and grid[r][c + span_c] == '[[H]]':
                    span_c += 1
                while r + span_r < num_rows and grid[r + span_r][c] == '[[V]]':
                    span_r += 1
                x0 = table.x_edges[c] + 1
                x1 = table.x_edges[c + span_c] - 1
                y0 = table.y_edges[r] + 1
                y1 = table.y_edges[r + span_r] - 1
                cell_crop = orig_bgr[y0:y1, x0:x1]
                grid[r][c] = ocr_single_cell(cell_crop, is_bold=is_row_bold)
                
        lines = ['| ' + ' | '.join(grid[0]) + ' |']
        lines.append('| ' + ' | '.join(['---'] * num_cols) + ' |')
        for r in range(1, num_rows):
            lines.append('| ' + ' | '.join(grid[r]) + ' |')
        table_blocks.append(chr(10).join(lines))
        
    return (chr(10) + chr(10)).join(table_blocks).strip() + chr(10) if table_blocks else EMPTY_TABLE


## 6. Chạy thử trên mẫu tập Train & So sánh Ground Truth


In [ ]:
# Kiểm tra thử trên 1 mẫu train
test_img_path = TRAIN_DIR / 'images' / 'train-00000_p01.jpg'
test_label_path = TRAIN_DIR / 'labels' / 'train-00000.md'

print('--- ĐANG TRÍCH XUẤT THỬ train-00000 ---')
pred_md = extract_table_from_image(test_img_path)
print(pred_md)

if test_label_path.exists():
    print('--- NHÃN CHUẨN GROUND TRUTH ---')
    print(test_label_path.read_text(encoding='utf-8'))


## 7. Sinh dự đoán toàn bộ tập Kiểm tra (Public / Private Test)


In [ ]:
from tqdm import tqdm

SPLIT_DIR = DATA / SPLIT
PREDICTIONS_DIR = RUNS / f'predictions_table_model_{SPLIT}'

records = load_manifest(SPLIT_DIR)
print(f'[dự đoán] {len(records)} tài liệu trong {SPLIT}')

outputs = []
started = time.time()

for record in tqdm(records, desc=f'Dự đoán {SPLIT}'):
    page_outputs = []
    for relative in record['image_paths']:
        img_path = SPLIT_DIR / relative
        md = extract_table_from_image(img_path)
        if md != EMPTY_TABLE:
            page_outputs.append(md.strip())
    final_md = ('\n\n'.join(page_outputs).strip() + '\n') if page_outputs else EMPTY_TABLE
    outputs.append(final_md)

write_predictions(PREDICTIONS_DIR, records, outputs)
print(f'[dự đoán] Hoàn thành {len(outputs)} tệp Markdown | Tổng thời gian: {time.time() - started:.1f}s')


## 8. Đóng gói file nộp bài submission.zip


In [ ]:
# Đóng gói file nộp theo đúng chuẩn BTC:
# - Lưu tại runs/submission_{SPLIT}.zip
# - Lưu file submission.zip ngay tại gốc thư mục tacvu4_table_extraction để tiện nộp
SUBMISSION_PATH = RUNS / f'submission_{SPLIT}.zip'
ROOT_SUBMISSION = ROOT / 'submission.zip'

make_predictions_zip(PREDICTIONS_DIR, SUBMISSION_PATH)
make_predictions_zip(PREDICTIONS_DIR, ROOT_SUBMISSION)

print('[nộp bài] Đã đóng gói thành công!')
print(f'  -> {SUBMISSION_PATH}')
print(f'  -> {ROOT_SUBMISSION}')
print(f'[kiểm tra] Số file .md trong gói: {len(list(PREDICTIONS_DIR.glob("*.md")))}')
